In [1]:
import sys
import os
import pandas as pd
import numpy as np

# --- Setup Paths ---
current_dir = os.getcwd()
sys.path.append(os.path.join(current_dir, 'OligoAI'))

In [3]:
from rinalmo.data.alphabet import Alphabet
from rinalmo.data.downstream.aso.dataset import ASODataset
from run_inference import run_inference

# --- Configuration ---
FLANK = 50
SEED = 1
BATCH_SIZE = 1000
NUM_WORKERS = 8 # Keeps the RTX 5090 fully saturated
MODEL_PATH = "OligoAI/OligoAI_11_09_25.ckpt"
ALPHABET = Alphabet()

raw_data_path = f"OligoAI/data/aso_inhibitions_21_08_25_incl_context_w_flank_{FLANK}_df.csv.gz"

print(f"\n{'='*60}")
print("1. Loading raw dataset to extract PSD3...")
print(f"{'='*60}")
df_raw = pd.read_csv(raw_data_path)

# Identify PSD3 by searching the custom_id column
psd3_mask = df_raw['target_gene'].str.contains('PSD3', case=False, na=False)

df_psd3 = df_raw[psd3_mask].copy()
df_main = df_raw[~psd3_mask].copy()

print(f"Total rows in raw data: {len(df_raw)}")
print(f"PSD3 rows extracted:    {len(df_psd3)}")
print(f"Main rows remaining:    {len(df_main)}")

# Save to temporary files for the dataset loaders
psd3_path = "OligoAI/data/temp_psd3_only.csv"
main_path = "OligoAI/data/temp_main_no_psd3.csv"

# Add a dummy 'split' column to PSD3 so the inference script stats don't error out
df_psd3['split'] = 'external_test'

df_psd3.to_csv(psd3_path, index=False)
df_main.to_csv(main_path, index=False)

# ==========================================
# PART A: PSD3 EXTERNAL VALIDATION
# ==========================================
print(f"\n{'='*60}")
print("PART A: Running Inference on Held-Out PSD3 Set")
print(f"{'='*60}")

results_psd3 = run_inference(
    model_checkpoint_path=MODEL_PATH,
    data_path=psd3_path,
    device="cuda",
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS
)

# ==========================================
# PART B: MAIN TEST SET (SEED 1)
# ==========================================
print(f"\n{'='*60}")
print(f"PART B: Processing Main Dataset (Seed {SEED})")
print(f"{'='*60}")

# Load the remaining data via ASODataset to perform the proper patent split
dataset_main = ASODataset(main_path, alphabet=ALPHABET, pad_to_max_len=False)
dataset_main.train_val_test_split(train_ratio=0.8, val_ratio=0.1, random_state=SEED)

# Extract the test split
split_csv_path = main_path.replace(".csv", ".withsplit.csv")
full_main_df = pd.read_csv(split_csv_path)

test_main_df = full_main_df[full_main_df['split'] == 'test']
test_main_path = "OligoAI/data/temp_main_test_seed1.csv"
test_main_df.to_csv(test_main_path, index=False)

print(f"\nRunning inference on Main Test Set (Seed {SEED})...")
results_main = run_inference(
    model_checkpoint_path=MODEL_PATH,
    data_path=test_main_path,
    device="cuda",
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS
)

# ==========================================
# PART C: METRICS EVALUATION
# ==========================================
print(f"\n{'='*60}")
print("FINAL BENCHMARK EVALUATION (Threshold: >= 3 samples per group)")
print(f"{'='*60}")

def evaluate_groups(df, name):
    correlations = []
    for custom_id in df['custom_id'].unique():
        group_df = df[df['custom_id'] == custom_id]
        
        # Using the >= 3 samples threshold
        if len(group_df) >= 3:
            if group_df['inhibition_percent'].nunique() > 1:
                corr, _ = spearmanr(group_df['predicted_inhibition_percent'], group_df['inhibition_percent'])
                if not np.isnan(corr):
                    correlations.append(corr)
    
    if correlations:
        med_sp = np.median(correlations)
        mean_sp = np.mean(correlations)
        print(f"--- {name} ---")
        print(f"Total Valid Groups Evaluated: {len(correlations)}")
        print(f"Median Spearman: {med_sp:.4f}")
        print(f"Mean Spearman:   {mean_sp:.4f}\n")
    else:
        print(f"--- {name} ---")
        print("Not enough data to calculate correlations.\n")

evaluate_groups(results_psd3, "Held-Out PSD3 Target")
evaluate_groups(results_main, "Main Test Set (No PSD3, Seed 1)")

# ==========================================
# CLEANUP
# ==========================================
# Remove the generated temp files to prevent drive bloat
files_to_remove = [
    psd3_path, 
    main_path, 
    split_csv_path, 
    test_main_path,
    psd3_path.replace('.csv', '.with_predictions.csv'),
    test_main_path.replace('.csv', '.with_predictions.csv')
]

for f in files_to_remove:
    if os.path.exists(f):
        try:
            os.remove(f)
        except Exception:
            pass
            
print("Temporary files cleaned up.")


1. Loading raw dataset to extract PSD3...
Total rows in raw data: 178624
PSD3 rows extracted:    3500
Main rows remaining:    175124

PART A: Running Inference on Held-Out PSD3 Set
Using device: cuda

Dataset split distribution:
split
external_test    3500
Name: count, dtype: int64
Loading model from: OligoAI/OligoAI_11_09_25.ckpt


/home/michael/anaconda3/envs/oligo_5090_hybrid/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'scaler' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['scaler'])`.


Model loaded successfully with saved scaler: <class 'rinalmo.utils.scaler.StandardScaler'>
Using Transfection Method Vocabulary: {'Electroporation': 0, 'Gymnosis': 1, 'Other': 2, 'Lipofection': 3}
Using Chemistry Vocabulary: {'<pad>': 0, 'DNA': 1, 'MOE': 2, 'cET': 3}
Using Backbone Vocabulary: {'<pad>': 0, 'PO': 1, 'PS': 2}
Loaded dataset with 3500 samples

Starting inference...
Inference completed. Generated 3500 predictions

Overall Prediction Statistics:
MAE: 23.8088
RMSE: 28.4722
R²: 0.1750
Pearson correlation: 0.4183
Spearman correlation: 0.4033

Per-split Statistics:
------------------------------------------------------------

Group-level Analysis:
Mean Spearman correlation across 5 custom_ids: 0.4013
Top pred target ratio (median): 1.2401 across 5 custom_ids

Predictions saved to: OligoAI/data/temp_psd3_only.with_predictions.csv

PART B: Processing Main Dataset (Seed 1)
Using Transfection Method Vocabulary: {'Electroporation': 0, 'Gymnosis': 1, 'Other': 2, 'Lipofection': 3}
Usi

/home/michael/anaconda3/envs/oligo_5090_hybrid/lib/python3.11/site-packages/lightning/pytorch/utilities/parsing.py:210: Attribute 'scaler' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['scaler'])`.


Model loaded successfully with saved scaler: <class 'rinalmo.utils.scaler.StandardScaler'>
Using Transfection Method Vocabulary: {'Electroporation': 0, 'Gymnosis': 1, 'Other': 2, 'Lipofection': 3}
Using Chemistry Vocabulary: {'<pad>': 0, 'DNA': 1, 'MOE': 2, 'cET': 3}
Using Backbone Vocabulary: {'<pad>': 0, 'PO': 1, 'PS': 2}
Loaded dataset with 22405 samples

Starting inference...
Processed 10000 samples...
Processed 20000 samples...
Inference completed. Generated 22405 predictions

Overall Prediction Statistics:
MAE: 21.1312
RMSE: 25.4893
R²: 0.2058
Pearson correlation: 0.4537
Spearman correlation: 0.4497

Per-split Statistics:
------------------------------------------------------------
TEST  - Samples:  22405
        MAE: 21.1312, RMSE: 25.4893
        R²: 0.2058, Pearson: 0.4537, Spearman: 0.4497
------------------------------------------------------------

Group-level Analysis:
Mean Spearman correlation across 297 custom_ids: 0.4029
Top pred target ratio (median): 1.4012 across 297

In [ ]:
import sys
import os
import math
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import spearmanr

# --- Setup Paths ---
current_dir = os.getcwd()
sys.path.append(os.path.join(current_dir, 'OligoAI'))
from run_inference import run_inference

# Use the exact path you provided
raw_data_path = Path('/home/michael/career/tauso_article/tauso_source2/notebooks/data/OligoAI/raw_data/aso_inhibitions_with_canonical_gene_processed_averaged.csv.gz')
model_path = "OligoAI/OligoAI_11_09_25.ckpt"

print(f"Loading master dataset from {raw_data_path}...")
df_master = pd.read_csv(raw_data_path)

# Ensure split column exists
if 'split' not in df_master.columns:
    raise ValueError("CRITICAL ERROR: 'split' column not found in the dataset!")

# --- Rename Schema ---
rename_back_scheme = {
    "Sequence": "aso_sequence_5_to_3",
    "Cell_line": "cell_line",
    "Cell line organism": "cell_line_species",
    "Inhibition(%)": "inhibition_percent",
    "ASO_volume(nM)": "dosage",
}

# --- Metric Helpers ---
def calculate_median_spearman(res_df):
    correlations = []
    for custom_id, group in res_df.groupby('custom_id'):
        if len(group) >= 3 and group['inhibition_percent'].nunique() > 1:
            corr, _ = spearmanr(group['predicted_inhibition_percent'], group['inhibition_percent'])
            if not np.isnan(corr):
                correlations.append(corr)
    return np.median(correlations) if correlations else np.nan, len(correlations)

def print_psd3_top1_metrics(res_df, split_name):
    # Isolate PSD3 (checking either target_gene or the explicit patent ID)
    if 'target_gene' in res_df.columns:
        psd3_mask = res_df['target_gene'].str.contains('PSD3', case=False, na=False)
    else:
        psd3_mask = res_df['custom_id'].str.contains('US20230167446A1', case=False, na=False)
        
    psd3_df = res_df[psd3_mask]
    
    if psd3_df.empty:
        print(f"  -> No PSD3 data found in {split_name.upper()} split.")
        return

    print(f"  -> PSD3 data found! Calculating Top 1% per table...")
    for custom_id, group in psd3_df.groupby('custom_id'):
        # math.ceil ensures we always grab at least 1 ASO even for small tables
        k_top = max(1, math.ceil(len(group) * 0.01))
        top_preds = group.nlargest(k_top, 'predicted_inhibition_percent')
        median_actual = top_preds['inhibition_percent'].median()
        
        short_name = str(custom_id).split('/')[-1]
        print(f"     Table: {short_name:35} | Total ASOs: {len(group):4} | Top 1% Selected: {k_top:2} | Median Top 1% Actual: {median_actual:.2f}%")


# --- Execution Pipeline ---
splits_to_process = ['test']
results_dict = {}

for split_name in splits_to_process:
    print(f"\n{'='*80}")
    print(f"PROCESSING SPLIT: {split_name.upper()}")
    print(f"{'='*80}")
    
    # 1. Isolate Split
    df_split = df_master[df_master['split'] == split_name].copy()
    if df_split.empty:
        print(f"No data found for split '{split_name}'. Skipping.")
        continue
        
    print(f"Found {len(df_split)} samples for {split_name}.")
    
    # 2. Rename Columns
    df_split = df_split.rename(columns=rename_back_scheme)
    
    # 3. Save Temp File
    temp_path = raw_data_path.parent / f"temp_{split_name}_inference.csv"
    df_split.to_csv(temp_path, index=False)
    
    # 4. Run Inference
    print(f"Running Inference on {split_name} set...")
    results_df = run_inference(
        model_checkpoint_path=model_path,
        data_path=str(temp_path),
        device="cuda",
        batch_size=100,
        num_workers=8
    )
    results_dict[split_name] = results_df
    
    # 5. Calculate Metrics
    print(f"\n--- METRICS FOR {split_name.upper()} ---")
    med_spearman, valid_groups = calculate_median_spearman(results_df)
    print(f"1. Median Spearman: {med_spearman:.4f} (Calculated across {valid_groups} evaluable custom_id groups)")
    
    print(f"\n2. PSD3 Top 1% Analysis:")
    print_psd3_top1_metrics(results_df, split_name)
    
    # Cleanup
    if temp_path.exists():
        temp_path.unlink()

print("\n" + "="*80)
print("ALL DONE. TEMPORARY FILES CLEANED UP.")
print("="*80)